In [ ]:
import pandas as pd

url = 'https://docs.google.com/spreadsheets/d/1g1aQ61vijh6uHJuc8sijeBEMsoIQ2a5yLwUK04Wptlg/export?format=csv';

dataset = pd.read_csv(url);

# Link para o código Colab: https://colab.research.google.com/drive/1p8GaeCKli_EzoCnQwKFbJ7zXmJ1ihzVs?usp=sharing

In [ ]:
# 🔹 Mapeamento CORRETO (usando nomes completos das colunas)
COLUMN_MAP = {
    'Você ficou gripado no ano passado ?': 'ficou_gripado',
    'Você tomou vacina da gripe no ano passado?': 'vacina_gripe',
    '  Você frequentou no ano passado,  semanalmente ambientes com muitas pessoas? (salas cheias, ônibus, eventos, etc.)  ': 'frequentou_ambientes_cheios',
    '  Você viajou no ano passado mais de 100 km de distância?  ': 'viajou_100km',
    '  Você tem alergia nas vias aéreas (rinite, sinusite, etc.)?  ': 'alergia_vias_aereas',
    'Quantas horas você dormiu em média por noite no ano passado?': 'horas_sono',
    'Você praticou atividade física no ano passado?': 'atividade_fisica',
    'Você se alimentou de forma balanceada no ano passado?': 'alimentacao_balanceada',
    'Em média, quantas vezes você lavou as mãos por dia no ano passado?': 'lavou_maos_por_dia',
    'Na sua percepção, o seu nível de estresse no ano passado foi:': 'nivel_estresse',
}

# 🔹 Renomear dataset
ds_maped = dataset.rename(columns=COLUMN_MAP)

ds_maped["nivel_estresse"] = pd.cut(
    ds_maped["nivel_estresse"],
    bins=[0, 2, 4, 5],
    labels=["baixo", "medio", "alto"]
)

# 🔹 Definição de features
FEATURES = {
  "risk_exposure_score": ["frequentou_ambientes_cheios","viajou_100km",],
  "protection_score": ["vacina_gripe","lavou_maos_por_dia",],
  "health_score": ["atividade_fisica","alimentacao_balanceada","horas_sono",],
  "respiratory_vulnerability": ["alergia_vias_aereas","nivel_estresse",],
  "preventive_behavior_score": ["vacina_gripe","lavou_maos_por_dia","atividade_fisica","alimentacao_balanceada",],
  "lifestyle_balance": ["horas_sono","atividade_fisica","alimentacao_balanceada","nivel_estresse",],
  "immunity_behavior_index": ["horas_sono","alimentacao_balanceada","atividade_fisica","nivel_estresse",],
}

TARGET = 'ficou_gripado'

In [ ]:
import pandas as pd
from collections import Counter

def prism(df, features, target):
    rules = []

    classes = df[target].unique()

    for c in classes:
        df_c = df.copy()

        while len(df_c[df_c[target] == c]) > 0:
            rule = []
            subset = df_c.copy()

            available_features = features.copy()

            while True:
                best_feature = None
                best_value = None
                best_prob = 0

                for f in available_features:
                    for v in subset[f].unique():
                        temp = subset[subset[f] == v]

                        if len(temp) == 0:
                            continue

                        prob = len(temp[temp[target] == c]) / len(temp)

                        if prob > best_prob:
                            best_prob = prob
                            best_feature = f
                            best_value = v

                if best_feature is None:
                    break

                rule.append((best_feature, best_value))

                subset = subset[subset[best_feature] == best_value]

                available_features.remove(best_feature)

                if len(subset[subset[target] != c]) == 0:
                    break

                if len(available_features) == 0:
                    break

            rules.append((rule, c))

            # FIX: Initialize mask with df_c's index to ensure alignment
            mask = pd.Series(True, index=df_c.index)
            for f, v in rule:
                mask &= (df_c[f] == v)

            df_c = df_c[~mask]

    return rules

def predict_prism(rules, sample):
    matching_predictions = []
    for conditions, label in rules:
        match = True
        for f, v in conditions:
            if sample.get(f) != v:
                match = False
                break
        if match:
            matching_predictions.append(label)
    
    if not matching_predictions:
        return None  # No rule matched
    
    # Use majority vote among matching rules
    prediction_counts = Counter(matching_predictions)
    # Return the most common prediction. If there's a tie, Counter.most_common() picks the first one encountered.
    return prediction_counts.most_common(1)[0][0]


In [ ]:
from sklearn.metrics import accuracy_score

def predict_df_prism(rules, df):
    preds = []

    for _, row in df.iterrows():
        pred = predict_prism(rules, row.to_dict())
        preds.append(pred)

    return preds


train = ds_maped.iloc[:150]
test = ds_maped.iloc[150:]

# Ensure feature_cols uses 'risk_exposure_score' as requested
feature_cols = FEATURES["risk_exposure_score"]
rules = prism(train, feature_cols, TARGET)

y_pred = predict_df_prism(rules, test)
y_true = test[TARGET]

# remove None
valid = [(yt, yp) for yt, yp in zip(y_true, y_pred) if yp is not None]

y_true_valid, y_pred_valid = zip(*valid)

print("Acurácia:", accuracy_score(y_true_valid, y_pred_valid))